# Lab 1 · Baseline CNN pada CIFAR-10

Terkait: **Bab 01 · Memahami Sistem ML/DL Secara Praktis**

## Tujuan
1. Memverifikasi template repo dapat dijalankan end-to-end (dry-run lolos).
2. Melatih baseline `SimpleCNN` pada CIFAR-10 selama 30 epoch.
3. Memplot kurva loss dan akurasi per epoch.
4. Menghitung confusion matrix pada test set.
5. Menganalisis 10 prediksi paling salah-yakin.

## Checklist
- [ ] `python -m src.train --config configs/baseline.yaml --dry-run` lolos.
- [ ] Training penuh (30 epoch) selesai; `val_acc > 0.70` untuk CIFAR-10.
- [ ] Kurva loss dan akurasi diplot dari log CSV.
- [ ] Confusion matrix divisualisasikan.
- [ ] 10 sampel paling confident-salah diinspeksi dan dijelaskan.
- [ ] Tiga pertanyaan refleksi dijawab.

## 0. Setup

Jalankan dari folder `template_repo/`. Pastikan environment sudah aktif.

### Lokal
```bash
pip install -e .
```

### Google Colab
Jalankan sel kode **0a. Setup Colab** di bawah ini. Setelah selesai:
- **Working directory** otomatis: `/content/ModulePembelajaran/ModulePembelajaran/template_repo`
- Semua perintah `configs/baseline.yaml`, `experiments/`, `src.` akan pakai path ini
- Lanjut ke **1. Periksa Arsitektur**

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/muhammad-zainal-muttaqin/ModulePembelajaran.git'
    REPO_DIR = '/content/ModulePembelajaran'

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        print('Repo sudah ada, skip clone.')

    # Pindah ke template_repo (semua lab dijalankan dari sini)
    %cd {REPO_DIR}/ModulePembelajaran/template_repo

    # Install dependencies
    !pip install -e .
    print('\nSetup Colab selesai. Working dir:', os.getcwd())
else:
    print('Environment lokal. Lewati setup Colab.')

In [ ]:
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

## 1. Periksa arsitektur sebelum training

In [ ]:
from src.models import SimpleCNN
from src.utils import count_parameters

model = SimpleCNN(num_classes=10)
print(model)
print(f'\ntrainable params: {count_parameters(model):,}')

# Verifikasi forward pass - input CIFAR-10: (B, 3, 32, 32)
dummy = torch.randn(4, 3, 32, 32)
out = model(dummy)
print(f'input shape: {dummy.shape} → output shape: {out.shape}')  # harus (4, 10)

## 2. Smoke test (dry-run)

Jalankan smoke test sebelum training penuh — <1 menit CPU, 1 epoch, 100 sampel.

**Jika gagal di sini, perbaiki dulu sebelum lanjut.**

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'src.train', '--config', 'configs/baseline.yaml', '--dry-run'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
assert result.returncode == 0, 'Dry-run gagal! Perbaiki dulu.'

## 3. Training penuh

Training lebih cepat dari terminal daripada dalam notebook. Jalankan:

### Lokal
```bash
python -m src.train --config configs/baseline.yaml --seed 42
```

### Google Colab — dari sel notebook baru
```python
!python -m src.train --config configs/baseline.yaml --seed 42
```

### Google Colab — dari terminal (klik ⋮ → Connect → Connect to terminal)
```bash
cd /content/ModulePembelajaran/ModulePembelajaran/template_repo && python -m src.train --config configs/baseline.yaml --seed 42
```

Tunggu selesai (≈10-20 menit CPU, <5 menit GPU), lalu lanjutkan ke **4. Baca hasil**.

## 4. Baca hasil dan plot kurva training

In [ ]:
import json
from pathlib import Path

exp_dir = Path('experiments/baseline_seed42')
summary = json.loads((exp_dir / 'summary.json').read_text())
print('Summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Baca metrics dari log (parse train.log untuk per-epoch data)
import re

log_path = exp_dir / 'train.log'
epochs_data = []
pattern = re.compile(
    r'epoch=(\d+) train_loss=([\d.]+) train_acc=([\d.]+) val_loss=([\d.]+) val_acc=([\d.]+)'
)
for line in log_path.read_text(encoding='utf-8').splitlines():
    m = pattern.search(line)
    if m:
        epochs_data.append({
            'epoch': int(m.group(1)),
            'train_loss': float(m.group(2)),
            'train_acc': float(m.group(3)),
            'val_loss': float(m.group(4)),
            'val_acc': float(m.group(5)),
        })

df_log = pd.DataFrame(epochs_data)
print(f'Epoch terbaca: {len(df_log)}')
df_log.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot loss
axes[0].plot(df_log['epoch'], df_log['train_loss'], label='train', color='steelblue')
axes[0].plot(df_log['epoch'], df_log['val_loss'], label='val', color='tomato')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss per Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot akurasi
axes[1].plot(df_log['epoch'], df_log['train_acc'] * 100, label='train', color='steelblue')
axes[1].plot(df_log['epoch'], df_log['val_acc'] * 100, label='val', color='tomato')
axes[1].axhline(70, color='gray', linestyle='--', alpha=0.5, label='target 70%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy per Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(exp_dir / 'loss_acc_curve.png', dpi=120, bbox_inches='tight')
plt.show()

# Deteksi overfitting
last_row = df_log.iloc[-1]
gap = last_row['train_acc'] - last_row['val_acc']
print(f'\nGap train-val akurasi di epoch terakhir: {gap*100:.1f}%')
if gap > 0.15:
    print('⚠ Gap > 15% — indikasi overfitting. Coba: augmentasi lebih kuat, weight decay, dropout.')
elif gap < 0.02:
    print('⚠ Gap < 2% — mungkin underfitting atau dataset terlalu mudah.')
else:
    print('✓ Gap wajar.')

## 5. Confusion matrix pada test set

In [ ]:
import numpy as np
from src.models import build_model
from src.utils import load_config
from src.data import build_loaders

cfg = load_config('configs/baseline.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = build_model(cfg['model']).to(device)
ckpt = torch.load(exp_dir / 'ckpt_best.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print('Loaded checkpoint - epoch:', ckpt['meta']['epoch'], '| val_acc:', ckpt['meta']['metric_value'])

In [ ]:
_, _, test_loader = build_loaders(cfg['data'])

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = torch.as_tensor(y).long().view(-1)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_acc = (all_preds == all_labels).mean()
print(f'Test accuracy: {test_acc*100:.2f}%')

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=CLASSES, yticklabels=CLASSES,
    vmin=0, vmax=1, ax=ax
)
ax.set_xlabel('Prediksi')
ax.set_ylabel('Label Asli')
ax.set_title('Confusion Matrix (normalized) - Test Set')
plt.tight_layout()
plt.savefig(exp_dir / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# Cetak per-class accuracy
print('\nPer-class accuracy:')
for i, cls in enumerate(CLASSES):
    print(f'  {cls:12s}: {cm_norm[i, i]*100:.1f}%')

## 6. Analisis kesalahan: 10 prediksi paling confident-salah

Ini adalah langkah yang sering dilewati tapi sangat informatif: di mana model *paling yakin tapi salah*?

In [ ]:
# Confidence = probabilitas kelas yang diprediksi
pred_confidences = all_probs[np.arange(len(all_preds)), all_preds]

# Filter hanya yang salah
wrong_mask = all_preds != all_labels
wrong_indices = np.where(wrong_mask)[0]
wrong_confidences = pred_confidences[wrong_mask]

# Top-10 paling confident di antara yang salah
top10_idx = wrong_indices[np.argsort(wrong_confidences)[::-1][:10]]

print(f'Total prediksi salah: {wrong_mask.sum()} dari {len(all_labels)}')
print(f'\nTop-10 paling confident-salah:')
print(f'{"Idx":>6} {"True":>12} {"Pred":>12} {"Confidence":>12}')
for i in top10_idx:
    print(f'{i:>6} {CLASSES[all_labels[i]]:>12} {CLASSES[all_preds[i]]:>12} {pred_confidences[i]:>12.3f}')

In [ ]:
import torchvision

# Ambil gambar asli dari test dataset untuk divisualisasikan
test_dataset = test_loader.dataset

# Denormalisasi untuk visualisasi
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for plot_i, data_i in enumerate(top10_idx):
    img_tensor, _ = test_dataset[data_i]
    img = (img_tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax = axes[plot_i // 5][plot_i % 5]
    ax.imshow(img)
    true_cls = CLASSES[all_labels[data_i]]
    pred_cls = CLASSES[all_preds[data_i]]
    conf = pred_confidences[data_i]
    ax.set_title(f'True: {true_cls}\nPred: {pred_cls} ({conf:.2f})', fontsize=8)
    ax.axis('off')

plt.suptitle('10 Prediksi Paling Confident-Salah', fontsize=11)
plt.tight_layout()
plt.savefig(exp_dir / 'confident_errors.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Inspeksi checkpoint metadata

In [ ]:
print('Checkpoint metadata:')
for k, v in ckpt['meta'].items():
    print(f'  {k}: {v}')

print('\nConfig dalam checkpoint (sebagian):')
for section in ['model', 'loss', 'optim', 'scheduler']:
    print(f'  {section}: {ckpt["config"].get(section, {})}')

## 8. Refleksi

Jawab tiga pertanyaan ini di sel markdown baru di bawah. Tidak ada jawaban benar/salah — yang penting adalah alasan.

1. Kelas mana yang paling sering dikacaukan satu sama lain (lihat confusion matrix)? Secara visual, mengapa masuk akal? Apa property visual yang membuat keduanya sulit dibedakan?

2. Di antara 10 gambar confident-salah, apakah ada pola? Contoh: pencahayaan, angle, objek sebagian terhalang? Jika kamu adalah arsitektur, apa yang membuatmu "yakin" pada prediksi yang salah itu?

3. Jelaskan satu keputusan desain di `SimpleCNN` (lihat `src/models.py`) yang tidak kamu buat sendiri tapi sekarang kamu pahami setelah melihat hasilnya. Contoh: `AdaptiveAvgPool2d(1)`, urutan BatchNorm + ReLU, ukuran kernel pertama.

### Jawaban Refleksi

**1. Kelas yang sering dikacaukan:**

> *[tulis di sini]*

**2. Pola di gambar confident-salah:**

> *[tulis di sini]*

**3. Keputusan desain yang dipahami:**

> *[tulis di sini]*